# Wasabi Sync — Download Notebook

Download individual files **or** entire virtual folders from your Wasabi S3 bucket.

**Sections**
1. Install & imports  
2. Configuration  
3. Helper utilities (progress, listing)  
4. Download a single file  
5. Download a folder (prefix)  
6. Batch download from a list of keys

## 1 — Install & Imports

In [16]:
%pip install boto3 tqdm python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


In [17]:
import os
import sys
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from typing import Optional

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError
from tqdm.notebook import tqdm

# ── optional: load .env from the repo root ──────────────────────────────────
try:
    from dotenv import load_dotenv
    _env = Path(__file__).parents[1] / ".env" if "__file__" in dir() else Path("..") / ".env"
    load_dotenv(_env, override=False)
    print("✓ .env loaded")
except Exception:
    pass  # running without dotenv is fine

✓ .env loaded


## 2 — Configuration

Fill in your Wasabi credentials below **or** set them as environment variables / in a `.env` file at the repo root.

## 3 — Helper Utilities

In [18]:
def _env(name: str, default: str = "") -> str:
    """
    Read a config value: prefer the module-level variable (set in Section 2),
    fall back to the OS environment variable, then use *default*.
    This makes every helper cell order-independent.
    """
    return globals().get(name) or os.getenv(name, default)


def _bucket() -> str:
    """Return the active bucket name (module global or env var)."""
    return _env("WASABI_BUCKET", "my-bucket")


def get_client() -> boto3.client:
    """Return a boto3 S3 client pointed at Wasabi."""
    return boto3.client(
        "s3",
        endpoint_url=_env("WASABI_ENDPOINT", "https://s3.wasabisys.com"),
        region_name=_env("WASABI_REGION",   "us-east-1"),
        aws_access_key_id=_env("WASABI_ACCESS_KEY_ID"),
        aws_secret_access_key=_env("WASABI_SECRET_ACCESS_KEY"),
        config=Config(
            signature_version="s3v4",
            retries={"max_attempts": 3, "mode": "adaptive"},
        ),
    )


def test_connection(client=None) -> bool:
    """Verify credentials and bucket access. Returns True on success."""
    client = client or get_client()
    bucket = _bucket()
    try:
        client.head_bucket(Bucket=bucket)
        print(f"\u2713 Connected \u2014 bucket '{bucket}' is accessible")
        return True
    except ClientError as e:
        code = e.response["Error"]["Code"]
        print(f"\u2717 Connection failed ({code}): {e}")
        return False


def list_objects(prefix: str = "", client=None) -> list[dict]:
    """
    List all objects under *prefix* in the bucket.
    Returns a list of dicts with keys: key, size, last_modified, etag.
    """
    client = client or get_client()
    objects = []
    paginator = client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=_bucket(), Prefix=prefix):
        for obj in page.get("Contents", []):
            objects.append({
                "key":           obj["Key"],
                "size":          obj["Size"],
                "last_modified": obj["LastModified"].isoformat(),
                "etag":          obj["ETag"].strip('"'),
            })
    return objects


def _human_size(n_bytes: int) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n_bytes < 1024:
            return f"{n_bytes:.1f} {unit}"
        n_bytes /= 1024
    return f"{n_bytes:.1f} PB"


# ── Quick connectivity check ──────────────────────────────────────────────────
_client = get_client()
test_connection(_client)

✗ Connection failed (400): An error occurred (400) when calling the HeadBucket operation: Bad Request


False

## 4 — Download a Single File

In [19]:
def download_file(
    key: str,
    local_path: Optional[str | Path] = None,
    overwrite: bool = False,
    client=None,
) -> Path:
    """
    Download a single object from the bucket.

    Parameters
    ----------
    key        : S3 object key, e.g. "datasets/train/image_001.jpg"
    local_path : Destination path on disk. Defaults to
                 LOCAL_DOWNLOAD_ROOT / key  (mirrors bucket structure).
    overwrite  : Skip the download if the file already exists when False.
    client     : Reuse an existing boto3 client (optional).

    Returns the local Path of the downloaded file.
    """
    client = client or get_client()

    if local_path is None:
        local_path = globals().get("LOCAL_DOWNLOAD_ROOT", Path("./downloads")) / key
    local_path = Path(local_path)

    if local_path.exists() and not overwrite:
        print(f"⏭  Skipped (already exists): {local_path}")
        return local_path

    # Get the object size for the progress bar
    try:
        meta = client.head_object(Bucket=WASABI_BUCKET, Key=key)
        total_bytes = meta["ContentLength"]
    except ClientError:
        total_bytes = None

    local_path.parent.mkdir(parents=True, exist_ok=True)

    with tqdm(
        total=total_bytes,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        desc=Path(key).name,
        leave=True,
    ) as bar:
        def _progress(n_bytes):
            bar.update(n_bytes)

        client.download_file(
            WASABI_BUCKET,
            key,
            str(local_path),
            Callback=_progress,
        )

    size_str = _human_size(total_bytes) if total_bytes else "?"
    print(f"✓ Downloaded ({size_str}): {local_path}")
    return local_path


# ── Usage ─────────────────────────────────────────────────────────────────────
# Change OBJECT_KEY to the key of the file you want to download.
OBJECT_KEY = "datasets/sample.csv"   # ← edit this

downloaded = download_file(OBJECT_KEY)
print(downloaded)

NameError: name 'WASABI_BUCKET' is not defined

## 5 — Download a Folder (Prefix)

Downloads **all** objects that share a common prefix (virtual folder), preserving the remote path structure under `LOCAL_DOWNLOAD_ROOT`.

In [ ]:
@dataclass
class FolderDownloadResult:
    prefix:    str
    total:     int = 0
    succeeded: int = 0
    skipped:   int = 0
    failed:    list[str] = field(default_factory=list)

    def __str__(self):
        return (
            f"Prefix  : {self.prefix}\n"
            f"Total   : {self.total}\n"
            f"✓ Done  : {self.succeeded}\n"
            f"⏭ Skip  : {self.skipped}\n"
            f"✗ Failed: {len(self.failed)}"
            + (f"\n  {chr(10).join(self.failed)}" if self.failed else "")
        )


def download_folder(
    prefix: str,
    local_root: Optional[str | Path] = None,
    overwrite: bool = False,
    strip_prefix: bool = True,
    workers: int = MAX_WORKERS,
    client=None,
) -> FolderDownloadResult:
    """
    Download every object whose key starts with *prefix*.

    Parameters
    ----------
    prefix       : S3 key prefix (virtual folder), e.g. "datasets/train/".
                   A trailing slash is added automatically if absent.
    local_root   : Local directory to save into. Defaults to
                   LOCAL_DOWNLOAD_ROOT / prefix.
    overwrite    : Re-download files that already exist when True.
    strip_prefix : When True, the prefix is stripped from local paths so that
                   "datasets/train/img.jpg" → "<local_root>/img.jpg".
                   When False, the full key path is mirrored.
    workers      : Number of parallel download threads.
    client       : Reuse an existing boto3 client.

    Returns a FolderDownloadResult summary.
    """
    client = client or get_client()

    # Normalise prefix
    if prefix and not prefix.endswith("/"):
        prefix = prefix + "/"

    if local_root is None:
        local_root = LOCAL_DOWNLOAD_ROOT / (prefix.rstrip("/") if strip_prefix else "")
    local_root = Path(local_root)

    print(f"Listing objects under prefix: '{prefix}' …")
    objects = list_objects(prefix=prefix, client=client)

    if not objects:
        print("No objects found under the given prefix.")
        return FolderDownloadResult(prefix=prefix)

    total_bytes = sum(o["size"] for o in objects)
    print(f"Found {len(objects)} object(s) — {_human_size(total_bytes)} total\n")

    result = FolderDownloadResult(prefix=prefix, total=len(objects))
    lock = threading.Lock()

    # Overall byte-level progress bar
    overall_bar = tqdm(
        total=total_bytes,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        desc="Overall",
        position=0,
    )

    def _download_one(obj: dict):
        key = obj["key"]
        rel = key[len(prefix):] if strip_prefix else key
        if not rel:          # skip the "directory placeholder" object itself
            return "skip", key
        dest = local_root / rel

        if dest.exists() and not overwrite:
            with lock:
                overall_bar.update(obj["size"])
            return "skip", key

        dest.parent.mkdir(parents=True, exist_ok=True)

        bytes_downloaded = [0]

        def _cb(n):
            bytes_downloaded[0] += n
            overall_bar.update(n)

        try:
            client.download_file(WASABI_BUCKET, key, str(dest), Callback=_cb)
            remaining = obj["size"] - bytes_downloaded[0]
            if remaining > 0:
                overall_bar.update(remaining)
            return "ok", key
        except Exception as exc:
            remaining = obj["size"] - bytes_downloaded[0]
            if remaining > 0:
                overall_bar.update(remaining)
            return "fail", f"{key} — {exc}"

    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(_download_one, obj): obj for obj in objects}
        for future in tqdm(
            as_completed(futures),
            total=len(objects),
            desc="Files",
            position=1,
            leave=True,
        ):
            status, info = future.result()
            with lock:
                if status == "ok":
                    result.succeeded += 1
                elif status == "skip":
                    result.skipped += 1
                else:
                    result.failed.append(info)

    overall_bar.close()
    print(f"\n{result}")
    return result


# ── Usage ─────────────────────────────────────────────────────────────────────
# Set FOLDER_PREFIX to the virtual folder you want to download.
# Examples:
#   "datasets/"           — entire top-level folder
#   "datasets/train/"     — specific sub-folder
#   "projects/my_proj/"   — a project folder
FOLDER_PREFIX = "datasets/"   # ← edit this

folder_result = download_folder(
    prefix=FOLDER_PREFIX,
    strip_prefix=True,   # set False to mirror full bucket path locally
    overwrite=False,     # set True to re-download existing files
)

## 6 — Batch Download from a Key List

Useful when you already know exactly which files you want across different folders.

In [ ]:
def batch_download(
    keys: list[str],
    local_root: Optional[str | Path] = None,
    overwrite: bool = False,
    workers: int = globals().get("MAX_WORKERS", 4),
    client=None,
) -> dict:
    """
    Download an explicit list of S3 keys in parallel.

    Parameters
    ----------
    keys       : List of full S3 object keys to download.
    local_root : Local root directory; keys are mirrored relative to it.
    overwrite  : Re-download if local file already exists.
    workers    : Parallel download threads.
    client     : Reuse an existing boto3 client.

    Returns a summary dict {succeeded, skipped, failed}.
    """
    client = client or get_client()
    local_root = Path(local_root) if local_root else globals().get("LOCAL_DOWNLOAD_ROOT", Path("./downloads"))
    summary = {"succeeded": 0, "skipped": 0, "failed": []}
    lock = threading.Lock()

    def _download_one(key: str):
        dest = local_root / key
        if dest.exists() and not overwrite:
            return "skip", key
        dest.parent.mkdir(parents=True, exist_ok=True)
        try:
            client.download_file(_bucket(), key, str(dest))
            return "ok", key
        except Exception as exc:
            return "fail", f"{key} — {exc}"

    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(_download_one, k): k for k in keys}
        for future in tqdm(as_completed(futures), total=len(keys), desc="Batch"):
            status, info = future.result()
            with lock:
                if status == "ok":
                    summary["succeeded"] += 1
                elif status == "skip":
                    summary["skipped"] += 1
                else:
                    summary["failed"].append(info)

    print(f"\n✓ {summary['succeeded']}  ⏭ {summary['skipped']}  ✗ {len(summary['failed'])}")
    if summary["failed"]:
        print("Failed keys:")
        for f in summary["failed"]:
            print(f"  {f}")
    return summary


# ── Usage ─────────────────────────────────────────────────────────────────────
KEYS_TO_DOWNLOAD = [
    "datasets/train/image_001.jpg",
    "datasets/train/image_002.jpg",
    "datasets/labels/train.csv",
]

batch_result = batch_download(KEYS_TO_DOWNLOAD)

## Appendix — Browse Bucket Contents

Run this cell at any time to list what's in your bucket (or under a specific prefix).